# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [1]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

Rozpoczynam pobieranie SEKWENCYJNE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Szalone nożyczki (Teatr Bagatela)
2. Atrament dla leworęcznych. Komedia absurdalna
3. Karty na stół
4. Nasze żony
5. Dizajn i emocje – jak działa na nas projektowanie?
6. Martwe natury
7. Bajki dla niegrzecznych
8. Hexvessel, Aluk Todolo, BaarRa w Zaścianku
9. Eros Ramazzotti: Una storia importante w Tauron Arenie Kraków
10. Wróg ludu

Czas wykonania: 3.01s


In [2]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Szalone nożyczki (Teatr Bagatela)
2. Atrament dla leworęcznych. Komedia absurdalna
3. Karty na stół
4. Nasze żony
5. Dizajn i emocje – jak działa na nas projektowanie?
6. Martwe natury
7. Bajki dla niegrzecznych
8. Hexvessel, Aluk Todolo, BaarRa w Zaścianku
9. Eros Ramazzotti: Una storia importante w Tauron Arenie Kraków
10. Wróg ludu

Czas wykonania (wątki): 0.91s


--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [ ]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [ ]:
import multiprocessing
import time
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        _ = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

Praca na 12 procesach (rdzeniach)...
Zakończono w czasie 0.66s.


---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [3]:
import functools
from typing import Any
import requests
import time
import concurrent.futures

CAT_API_URL = "https://catfact.ninja/fact"

def timer_decorator() -> Any:
    """Dekorator mierzący czas wykonania funkcji."""
    def decorator(func : Any) -> Any:
        @functools.wraps(func)
        def wrapper(*args : Any, **kwargs : Any) -> Any:
            start = time.time()
            result = func(*args, **kwargs)
            end = time.time()
            print(f"Czas wykonania: {end - start:.2f}s")
            return result
        return wrapper
    return decorator

def fetch_cat_fact() -> str:
    """Pobiera losową ciekawostkę o kotach."""
    response = requests.get(CAT_API_URL)
    if response.status_code == 200:
        return response.json().get('fact')
    return "Brak danych"

@timer_decorator()
def get_facts_sequential(count:int) -> list[str]:
    """Pobiera ciekawostki o kotach sekwencyjnie."""
    facts:list[str] = []
    for _ in range(count):
        facts.append(fetch_cat_fact())
    return facts

@timer_decorator()
def get_facts_threaded(count:int) -> list[str]:
    """Pobiera ciekawostki o kotach wielowątkowo."""
    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
        futures = [executor.submit(fetch_cat_fact) for _ in range(count)]
        return [future.result() for future in concurrent.futures.as_completed(futures)]
    
if __name__ == "__main__":
    print("Pobieranie ciekawostek o kotach SEKWENCYJNIE:")
    sequential_facts = get_facts_sequential(20)
    
    print("\nPobieranie ciekawostek o kotach WIELOWĄTKOWO:")
    threaded_facts = get_facts_threaded(20)

Pobieranie ciekawostek o kotach SEKWENCYJNIE:
Czas wykonania: 13.40s

Pobieranie ciekawostek o kotach WIELOWĄTKOWO:
Czas wykonania: 3.41s


### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [ ]:
import queue
import threading
import time

TOTAL_NUMBERS = 20

number_queue: queue.Queue[int|None] = queue.Queue(maxsize=8)
even_numbers: list[int] = []
odd_numbers: list[int] = []


def producer(q: queue.Queue[int|None], limit: int):
    for number in range(limit):
        q.put(number)
        print(f"[PRODUCENT] Dodano: {number}")
        time.sleep(0.03)

    q.put(None)
    q.put(None)
    print("[PRODUCENT] Koniec produkcji.")


def consumer_even(q: queue.Queue[int|None], bucket: list[int]):
    while True:
        value = q.get()

        if value is None:
            q.task_done()
            print("[KONSUMENT PARZYSTE] Zakończenie pracy.")
            break

        if value % 2 == 0:
            bucket.append(value)
            print(f"[KONSUMENT PARZYSTE] Pobrano: {value}")
        else:
            q.put(value)
            time.sleep(0.01)

        q.task_done()


def consumer_odd(q: queue.Queue[int|None], bucket: list[int]):
    while True:
        value = q.get()

        if value is None:
            q.task_done()
            print("[KONSUMENT NIEPARZYSTE] Zakończenie pracy.")
            break

        if value % 2 == 1:
            bucket.append(value)
            print(f"[KONSUMENT NIEPARZYSTE] Pobrano: {value}")
        else:
            q.put(value)
            time.sleep(0.01)

        q.task_done()


producer_thread = threading.Thread(target=producer, args=(number_queue, TOTAL_NUMBERS))
even_thread = threading.Thread(target=consumer_even, args=(number_queue, even_numbers))
odd_thread = threading.Thread(target=consumer_odd, args=(number_queue, odd_numbers))

producer_thread.start()
even_thread.start()
odd_thread.start()

producer_thread.join()
number_queue.join()
even_thread.join()
odd_thread.join()

print("\n--- Podsumowanie ---")
print(f"Liczby parzyste ({len(even_numbers)}): {sorted(even_numbers)}")
print(f"Liczby nieparzyste ({len(odd_numbers)}): {sorted(odd_numbers)}")
print(f"Razem przetworzono: {len(even_numbers) + len(odd_numbers)}")

[PRODUCENT] Dodano: 0
[KONSUMENT PARZYSTE] Pobrano: 0
[PRODUCENT] Dodano: 1
[KONSUMENT NIEPARZYSTE] Pobrano: 1
[PRODUCENT] Dodano: 2
[KONSUMENT PARZYSTE] Pobrano: 2
[PRODUCENT] Dodano: 3
[KONSUMENT NIEPARZYSTE] Pobrano: 3
[PRODUCENT] Dodano: 4
[KONSUMENT PARZYSTE] Pobrano: 4
[PRODUCENT] Dodano: 5
[KONSUMENT NIEPARZYSTE] Pobrano: 5
[PRODUCENT] Dodano: 6
[KONSUMENT PARZYSTE] Pobrano: 6
[PRODUCENT] Dodano: 7[KONSUMENT NIEPARZYSTE] Pobrano: 7

[PRODUCENT] Dodano: 8
[KONSUMENT PARZYSTE] Pobrano: 8
[PRODUCENT] Dodano: 9
[KONSUMENT NIEPARZYSTE] Pobrano: 9
[PRODUCENT] Dodano: 10
[KONSUMENT PARZYSTE] Pobrano: 10
[PRODUCENT] Dodano: 11
[KONSUMENT NIEPARZYSTE] Pobrano: 11
[PRODUCENT] Dodano: 12
[KONSUMENT PARZYSTE] Pobrano: 12
[PRODUCENT] Dodano: 13
[KONSUMENT NIEPARZYSTE] Pobrano: 13
[PRODUCENT] Dodano: 14[KONSUMENT PARZYSTE] Pobrano: 14

[PRODUCENT] Dodano: 15
[KONSUMENT NIEPARZYSTE] Pobrano: 15
[PRODUCENT] Dodano: 16
[KONSUMENT PARZYSTE] Pobrano: 16
[PRODUCENT] Dodano: 17
[KONSUMENT NIEPARZYST

### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [ ]:
import multiprocessing
import time
from lab2_functions import calculate_power_sum


def run_power_sum_demo() -> None:
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()

    limit = 10_000
    numbers = list(range(1, limit + 1))

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.map(calculate_power_sum, numbers)

    total_sum = sum(results)
    print(f"Policzono {len(results)} wartości funkcji calculate_power_sum(n).")
    print(f"Suma końcowa: {total_sum}")
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")


if __name__ == "__main__":
    run_power_sum_demo()

Praca na 12 procesach (rdzeniach)...
Policzono 10000 wartości funkcji calculate_power_sum(n).
Suma końcowa: 995207854197964442547664779149713094406225681179265198421228880588357806296714186703210526583761834998114384212288730396901480433958551919996123240783736004317692688469399493136990101249303140987388533721044806766065437006409106438027712620589642097820975739314540123609570363583485256201869594157997582865282767878851852797262927388912728290810611143129578917963952248558896652100510662230361518505000
Zakończono w czasie 0.27s.
